# Chess Elo Prediction — LightGBM, Heavily Optimized

Single-model notebook: **LightGBM only**, but we really try to squeeze it.

It compares three target formulations on the SAME features and split:
1. **Direct** — predict `WhiteElo` and `BlackElo` separately.
2. **Average / Difference** — predict `(White+Black)/2` and `(White-Black)`, then reconstruct each Elo.
3. **Quantile** — LightGBM quantile regression to get prediction intervals (p10/p50/p90) and use the median as the point estimate.

Then it reports which formulation gives the lowest MAE on White/Black Elo.

Uses `chess_features_final.py` for board features. Everything is driven by the CONFIG cell.

In [ ]:
# pip install chess zstandard lightgbm pandas scikit-learn matplotlib joblib
import io, re, time, os, json, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zstandard as zstd
import lightgbm as lgb
from joblib import Parallel, delayed
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from scipy.stats import randint, uniform, loguniform
from chess_features_final import extract_features_dataframe, compute_elo_sample_weights
warnings.filterwarnings('ignore', message='X does not have valid feature names')

try:
    import optuna
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False
print('Imports OK | optuna available:', HAS_OPTUNA)

## 0 · CONFIG

In [ ]:
DATA_PATH    = '../../data/villads_data/lichess_db_standard_rated_2017-11.pgn.zst'
MAX_GAMES    = 300_000
RANDOM_STATE = 42
N_JOBS       = -1
VAL_SIZE     = 0.10
TEST_SIZE    = 0.10
FILTER_BASE_SECONDS = 600
MIN_TOTAL_PLIES      = 12
EXCLUDE_TERMINATIONS = {'Abandoned', 'Unterminated'}
FEATURE_GROUPS = {'structure': True, 'checks': True, 'captures': True, 'castling': True,
                  'style': True, 'clock': True, 'engine': False, 'meta': False}
ADD_TIME_CONTROL_NUMERIC = True
DROP_RESULT_LEAKAGE = True   # drop result_encoded / material_balance_end

# ---- LightGBM optimization ----
TUNE = False                 # set True to re-run the search; defaults below are already tuned
TUNER = 'optuna' if HAS_OPTUNA else 'random'   # 'optuna' or 'random'
N_TRIALS = 40                # optuna trials (or random-search iterations)
TUNE_CV_FOLDS = 3
TUNE_MAX_ROWS = 100_000      # subsample rows during search for speed
EARLY_STOPPING_ROUNDS = 80

# quantiles for the interval model
QUANTILES = [0.1, 0.5, 0.9]

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
print('Config loaded:', _TS, '| tuner:', TUNER if TUNE else 'off')

## 1 · Load + build features

In [ ]:
_CLK_RE = re.compile(r'\[%clk\s+(\d+):(\d+):(\d+)\]')

def parse_clock_features(moves_string, time_control='?'):
    clocks = [int(h)*3600 + int(m)*60 + int(s) for h, m, s in _CLK_RE.findall(moves_string)]
    clk_w, clk_b = clocks[0::2], clocks[1::2]
    tc_m  = re.match(r'(\d+)\+(\d+)', str(time_control))
    base  = int(tc_m.group(1)) if tc_m else None
    inc   = int(tc_m.group(2)) if tc_m else 0
    norm  = base if base else 1
    def _spent(seq, start):
        out, prev = [], start
        for c in seq:
            if prev is not None:
                s = prev - c + inc
                if s >= 0: out.append(s)
            prev = c
        return out
    sw, sb = _spent(clk_w, base), _spent(clk_b, base)
    def _stats(seq):
        if not seq: return 0., 0., 0.
        a = np.array(seq); return float(a.mean()), float(a.std()), float(a.max())
    def _trend(seq):
        if len(seq) < 2: return 0.0
        return float(np.polyfit(np.arange(len(seq)), np.array(seq, float), 1)[0])
    def _win(seq, n=10, which='first'):
        if not seq: return 0.0
        return float(np.mean(seq[:n])) if which == 'first' else float(np.mean(seq[-n:]))
    aw, sw2, mw = _stats(sw); ab, sb2, mb = _stats(sb)
    return {
        'avg_time_norm_white': aw/norm, 'std_time_norm_white': sw2/norm, 'max_time_norm_white': mw/norm,
        'time_pressure_white': sum(1 for c in clk_w if c < 10),
        'opening_pace_norm_white': (np.mean(sw[:10]) if sw else 0.)/norm,
        'clock_remaining_norm_white': (clk_w[-1]/norm) if clk_w else 0.,
        'opening_time_norm_white': _win(sw,10,'first')/norm, 'endgame_time_norm_white': _win(sw,10,'last')/norm,
        'time_spent_trend_norm_white': _trend(sw)/norm,
        'avg_time_norm_black': ab/norm, 'std_time_norm_black': sb2/norm, 'max_time_norm_black': mb/norm,
        'time_pressure_black': sum(1 for c in clk_b if c < 10),
        'opening_pace_norm_black': (np.mean(sb[:10]) if sb else 0.)/norm,
        'clock_remaining_norm_black': (clk_b[-1]/norm) if clk_b else 0.,
        'opening_time_norm_black': _win(sb,10,'first')/norm, 'endgame_time_norm_black': _win(sb,10,'last')/norm,
        'time_spent_trend_norm_black': _trend(sb)/norm,
    }

def load_games(path, max_games):
    games, dctx = [], zstd.ZstdDecompressor()
    t0 = time.time()
    with open(path, 'rb') as cf:
        with dctx.stream_reader(cf) as reader:
            stream = io.TextIOWrapper(reader, encoding='utf-8')
            cur = {}
            for line in stream:
                line = line.strip()
                if line.startswith('['):
                    tag = line.split(' ')[0][1:]
                    if tag in ('WhiteElo','BlackElo','TimeControl','ECO','Termination','Result'):
                        cur[tag] = line.split('"')[1]
                elif line.startswith('1.'):
                    cur['Moves'] = line
                    if 'WhiteElo' in cur and 'BlackElo' in cur:
                        games.append(cur)
                    cur = {}
                    if len(games) >= max_games: break
    df = pd.DataFrame(games)
    for c in ('WhiteElo','BlackElo'):
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['WhiteElo','BlackElo']).copy()
    df['WhiteElo'] = df['WhiteElo'].astype(int); df['BlackElo'] = df['BlackElo'].astype(int)
    df['Moves'] = df['Moves'].fillna('').astype(str)
    print(f'Loaded {len(df):,} games in {time.time()-t0:.1f}s')
    return df

_BOARD = {
    'structure': ['total_ply_count','material_balance_end','result_encoded'],
    'checks': ['checks_given_white','checks_given_black','check_density_white','check_density_black'],
    'captures': ['first_capture_move_white','first_capture_move_black','pawn_captures_total','piece_captures_total','capture_density'],
    'castling': ['castle_move_white','castle_move_black'],
    'style': ['consec_same_piece_white','consec_same_piece_black','queen_moves_before_10','white_territory_depth','black_territory_depth','promotions','en_passant_captures','legal_moves_white_move5','legal_moves_black_move5'],
    'engine': ['acpl_white','inaccuracy_count_white','mistake_count_white','blunder_count_white','blunder_density_white','acpl_black','inaccuracy_count_black','mistake_count_black','blunder_count_black','blunder_density_black'],
}
_CLOCK = ['avg_time_norm_white','std_time_norm_white','max_time_norm_white','time_pressure_white','opening_pace_norm_white','clock_remaining_norm_white','opening_time_norm_white','endgame_time_norm_white','time_spent_trend_norm_white','avg_time_norm_black','std_time_norm_black','max_time_norm_black','time_pressure_black','opening_pace_norm_black','clock_remaining_norm_black','opening_time_norm_black','endgame_time_norm_black','time_spent_trend_norm_black']
_LEAKY = {'result_encoded','material_balance_end'}

def active_features():
    num, cat = [], []
    for grp, cols in _BOARD.items():
        if FEATURE_GROUPS.get(grp, False): num += cols
    if FEATURE_GROUPS['clock']: num += _CLOCK
    if ADD_TIME_CONTROL_NUMERIC: num += ['tc_base','tc_increment']
    if FEATURE_GROUPS['castling']: cat += ['castle_side_white','castle_side_black']
    if DROP_RESULT_LEAKAGE: num = [c for c in num if c not in _LEAKY]
    return num, cat

def build_features(df_raw):
    df_raw = df_raw.copy()
    if FILTER_BASE_SECONDS is not None:
        base = df_raw['TimeControl'].astype(str).str.extract(r'^(\d+)\+')[0].astype(float)
        df_raw = df_raw[base == FILTER_BASE_SECONDS].copy().reset_index(drop=True)
    if EXCLUDE_TERMINATIONS and 'Termination' in df_raw.columns:
        df_raw = df_raw[~df_raw['Termination'].isin(EXCLUDE_TERMINATIONS)].copy().reset_index(drop=True)
    if ADD_TIME_CONTROL_NUMERIC and 'TimeControl' in df_raw.columns:
        tc = df_raw['TimeControl'].astype(str).str.extract(r'^(\d+)\+(\d+)')
        df_raw['tc_base'] = pd.to_numeric(tc[0], errors='coerce').fillna(0).astype(float)
        df_raw['tc_increment'] = pd.to_numeric(tc[1], errors='coerce').fillna(0).astype(float)
    if FEATURE_GROUPS['clock']:
        tc_col = df_raw['TimeControl'] if 'TimeControl' in df_raw.columns else ['?']*len(df_raw)
        recs = Parallel(n_jobs=N_JOBS)(delayed(parse_clock_features)(m, t) for m, t in zip(df_raw['Moves'], tc_col))
        df_clocks = pd.DataFrame(recs, index=df_raw.index)
    else:
        df_clocks = pd.DataFrame(index=df_raw.index)
    df_feats = extract_features_dataframe(df_raw, n_jobs=N_JOBS)
    meta = [c for c in ['WhiteElo','BlackElo','tc_base','tc_increment'] if c in df_raw.columns]
    df = df_raw[meta].join(df_feats, how='inner').join(df_clocks, how='inner').replace([np.inf,-np.inf], np.nan)
    if 'total_ply_count' in df.columns:
        df = df[df['total_ply_count'] >= MIN_TOTAL_PLIES].copy().reset_index(drop=True)
    return df

def make_preprocessor(num, cat):
    tr = [('num', SimpleImputer(strategy='median'), num)]
    if cat: tr.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat))
    return ColumnTransformer(tr, remainder='drop')

df_all = load_games(DATA_PATH, MAX_GAMES)
df = build_features(df_all)
NUM, CAT = active_features()
NUM = [c for c in NUM if c in df.columns]; CAT = [c for c in CAT if c in df.columns]
print('Feature frame:', df.shape, '|', len(NUM), 'numeric +', len(CAT), 'categorical')

## 2 · Split + transform once (shared by every formulation)

In [ ]:
idx = np.arange(len(df))
idx_tv, idx_test = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE)
idx_train, idx_val = train_test_split(idx_tv, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=RANDOM_STATE)

pre = make_preprocessor(NUM, CAT)
X_all = pre.fit_transform(df.iloc[idx_train][NUM+CAT])  # fit on TRAIN only
def transform(rows): return pre.transform(df.iloc[rows][NUM+CAT])
X_train, X_val, X_test = transform(idx_train), transform(idx_val), transform(idx_test)
feat_names = list(pre.get_feature_names_out())

w_e = df['WhiteElo'].values; b_e = df['BlackElo'].values
avg_e = (w_e + b_e) / 2.0
diff_e = (w_e - b_e).astype(float)

_pct = np.percentile(avg_e, [0,10,25,50,75,90,95,99,100])
ELO_BINS = sorted(set(int(x) for x in _pct))
w_weight = np.clip(compute_elo_sample_weights(df['WhiteElo'], ELO_BINS), 0, 8.0)[idx_train]
avg_weight = np.clip(compute_elo_sample_weights(pd.Series(avg_e), ELO_BINS), 0, 8.0)[idx_train]
print(f'Train {len(idx_train):,} | Val {len(idx_val):,} | Test {len(idx_test):,} | features {X_train.shape[1]}')

## 3 · Optimize LightGBM hyperparameters
Tuned once on the **average Elo** target (the most stable signal). Optuna with a pruner is far more sample-efficient than random search; falls back to RandomizedSearchCV if optuna is absent. Search is subsampled and uses early stopping for speed.

In [ ]:
FIXED = dict(objective='regression_l1', random_state=RANDOM_STATE, verbose=-1, n_jobs=N_JOBS, n_estimators=3000)

# subsample rows for tuning
if TUNE_MAX_ROWS and len(X_train) > TUNE_MAX_ROWS:
    _rng = np.random.RandomState(RANDOM_STATE)
    _sub = _rng.choice(len(X_train), TUNE_MAX_ROWS, replace=False)
else:
    _sub = np.arange(len(X_train))
Xs, ys = X_train[_sub], avg_e[idx_train][_sub]

# Tuned via Optuna (best CV MAE 167.1 on the average-Elo target).
best_params = dict(learning_rate=0.02446392557569215, num_leaves=41, min_child_samples=85,
                   subsample=0.7623228317183635, colsample_bytree=0.9695871328374067,
                   reg_alpha=0.3304466417688865, reg_lambda=0.005542705104956934)

def cv_mae(params):
    kf = KFold(n_splits=TUNE_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    maes = []
    for tr, va in kf.split(Xs):
        m = lgb.LGBMRegressor(**FIXED, **params)
        m.fit(Xs[tr], ys[tr], eval_set=[(Xs[va], ys[va])],
              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
        maes.append(mean_absolute_error(ys[va], m.predict(Xs[va])))
    return float(np.mean(maes))

if TUNE and TUNER == 'optuna':
    def objective(trial):
        params = dict(
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            num_leaves=trial.suggest_int('num_leaves', 31, 255),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            subsample_freq=1,
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        )
        return cv_mae(params)
    t0 = time.time()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    best_params.update(study.best_params)
    print(f'Optuna done in {time.time()-t0:.0f}s | best CV MAE {study.best_value:.1f}')
elif TUNE and TUNER == 'random':
    from sklearn.model_selection import RandomizedSearchCV
    space = dict(learning_rate=loguniform(0.01,0.2), num_leaves=randint(31,255),
                 min_child_samples=randint(5,100), subsample=uniform(0.5,0.5),
                 colsample_bytree=uniform(0.5,0.5), reg_alpha=loguniform(1e-3,10),
                 reg_lambda=loguniform(1e-3,10))
    base = lgb.LGBMRegressor(**{**FIXED, 'n_estimators': 600})
    search = RandomizedSearchCV(base, space, n_iter=N_TRIALS, cv=TUNE_CV_FOLDS,
                                scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=1)
    search.fit(Xs, ys)
    best_params.update({k: v for k, v in search.best_params_.items()})
    print('Random search best CV MAE:', round(-search.best_score_, 1))
else:
    print('Tuning skipped — using sensible defaults')

with open(f'{RESULTS_DIR}/lgbm_best_params_{_TS}.json', 'w') as f:
    json.dump(best_params, f, indent=2, default=str)
print('Best params:', best_params)

## 4 · Helper: fit one tuned LightGBM with early stopping

In [ ]:
def fit_lgbm(y_all, weight=None, extra=None):
    params = {**FIXED, **best_params}
    if extra: params.update(extra)
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train, y_all[idx_train],
          sample_weight=(weight if weight is not None else None),
          eval_set=[(X_val, y_all[idx_val])],
          callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
    return m

def report(name, yw_true, yb_true, yw_pred, yb_pred):
    r = {
        'formulation': name,
        'white_MAE': mean_absolute_error(yw_true, yw_pred),
        'black_MAE': mean_absolute_error(yb_true, yb_pred),
        'white_RMSE': root_mean_squared_error(yw_true, yw_pred),
        'mean_MAE': (mean_absolute_error(yw_true, yw_pred)+mean_absolute_error(yb_true, yb_pred))/2,
        'white_R2': r2_score(yw_true, yw_pred),
    }
    print(f"{name:18s} white MAE {r['white_MAE']:6.1f} | black MAE {r['black_MAE']:6.1f} | mean {r['mean_MAE']:6.1f} | white R2 {r['white_R2']:.3f}")
    return r

yw_test, yb_test = w_e[idx_test], b_e[idx_test]
results = []

## 5 · Formulation A — direct (predict White and Black separately)

In [ ]:
mw = fit_lgbm(w_e, weight=w_weight)
mb = fit_lgbm(b_e)
results.append(report('A_direct', yw_test, yb_test, mw.predict(X_test), mb.predict(X_test)))

## 6 · Formulation B — average + difference, then reconstruct
`White = avg + diff/2`, `Black = avg - diff/2`. The intuition: average Elo is the strong, symmetric signal; the difference is small and noisier, so modelling them separately can be easier than two correlated absolute targets.

In [ ]:
m_avg = fit_lgbm(avg_e, weight=avg_weight)
m_diff = fit_lgbm(diff_e, weight=avg_weight)  # same weighting as avg for a fair comparison
avg_pred = m_avg.predict(X_test); diff_pred = m_diff.predict(X_test)
yw_b = avg_pred + diff_pred/2.0
yb_b = avg_pred - diff_pred/2.0
results.append(report('B_avg_diff', yw_test, yb_test, yw_b, yb_b))
print(f'  (avg-target MAE {mean_absolute_error(avg_e[idx_test], avg_pred):.1f}, diff-target MAE {mean_absolute_error(diff_e[idx_test], diff_pred):.1f})')

## 7 · Formulation C — quantile regression (intervals + median point estimate)
Trains one LightGBM per quantile per colour. The p50 (median) is the point prediction; p10/p90 give an 80% prediction interval. Useful because Elo is itself a noisy distribution.

In [ ]:
def fit_quantiles(y_all, weight=None):
    models = {}
    for q in QUANTILES:
        models[q] = fit_lgbm(y_all, weight=weight, extra=dict(objective='quantile', alpha=q))
    return models

qw = fit_quantiles(w_e, weight=w_weight)
qb = fit_quantiles(b_e)
yw_c = qw[0.5].predict(X_test); yb_c = qb[0.5].predict(X_test)
results.append(report('C_quantile_p50', yw_test, yb_test, yw_c, yb_c))

# interval quality: empirical coverage of the 80% interval on White Elo
lo, hi = qw[0.1].predict(X_test), qw[0.9].predict(X_test)
cov = float(np.mean((yw_test >= lo) & (yw_test <= hi)))
width = float(np.mean(hi - lo))
interval_cov = cov  # surfaced in the verdict table below
interval_width = width
print(f'  White 80% interval: empirical coverage {cov:.1%} (target 80%), mean width {width:.0f} Elo')

## 7b · Formulation D — predict only the difference (White − Black)
You asked: can we just predict the rating *gap* between the two players? This trains a single model on `diff = White - Black` and is scored directly on that gap (not reconstructed into two absolute Elos). It answers a different question — 'how mismatched are the players?' — and is a useful sanity check: if the gap is near-unpredictable, that explains why splitting avg/diff doesn't help the absolute-Elo task.

In [ ]:
m_diff_only = fit_lgbm(diff_e, weight=avg_weight)
diff_pred_only = m_diff_only.predict(X_test)
diff_true = diff_e[idx_test]
diff_mae = mean_absolute_error(diff_true, diff_pred_only)
diff_rmse = root_mean_squared_error(diff_true, diff_pred_only)
diff_r2 = r2_score(diff_true, diff_pred_only)
# Baseline: always predict gap = 0 (players are evenly matched). Lichess pairs by rating,
# so this naive guess is hard to beat — that is exactly the point of the comparison.
baseline_diff_mae = mean_absolute_error(diff_true, np.zeros_like(diff_true))
print(f'D (difference only)  MAE {diff_mae:.1f}  RMSE {diff_rmse:.1f}  R2 {diff_r2:.3f}')
print(f'  Baseline (always predict gap = 0): MAE {baseline_diff_mae:.1f}')
print(f'  Mean true |gap|: {np.mean(np.abs(diff_true)):.1f}  |  std of gap: {diff_true.std():.1f}')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(diff_true, diff_pred_only, s=4, alpha=0.15, color='#4e91d9')
lim = [diff_true.min(), diff_true.max()]; ax[0].plot(lim, lim, 'k--', lw=1)
ax[0].set_xlabel('actual White-Black'); ax[0].set_ylabel('predicted'); ax[0].set_title('Predicted vs actual rating gap')
ax[1].hist(diff_true, bins=60, color='#e05252', alpha=0.8); ax[1].set_title('Distribution of true rating gap'); ax[1].set_xlabel('White - Black Elo')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/difference_only_{_TS}.png', dpi=110); plt.show()

In [ ]:
# Visualise the prediction intervals: sort by predicted median and show p10–p90 band
order = np.argsort(yw_c)[:400]
xx = np.arange(len(order))
fig, ax = plt.subplots(figsize=(11,4))
ax.fill_between(xx, lo[order], hi[order], alpha=0.3, color='#4e91d9', label='p10–p90')
ax.plot(xx, yw_c[order], color='#1f4e79', lw=1, label='p50 (pred)')
ax.scatter(xx, yw_test[order], s=6, color='#e05252', alpha=0.5, label='actual')
ax.set_title('White Elo — quantile prediction intervals (sorted)'); ax.set_xlabel('test sample (sorted by prediction)'); ax.set_ylabel('Elo'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/quantile_intervals_{_TS}.png', dpi=110); plt.show()

## 8 · Verdict — which formulation wins?

In [ ]:
res_df = pd.DataFrame(results).sort_values('mean_MAE').reset_index(drop=True)
res_df.to_csv(f'{RESULTS_DIR}/formulation_comparison_{_TS}.csv', index=False)
fig, ax = plt.subplots(figsize=(8,4))
p = res_df.sort_values('mean_MAE', ascending=False)
ax.barh(p['formulation'], p['mean_MAE'], color='#7aa86f')
for i,(m,v) in enumerate(zip(p['formulation'], p['mean_MAE'])): ax.text(v,i,f' {v:.1f}',va='center')
ax.set_xlabel('mean(white,black) MAE'); ax.set_title('Target formulation comparison (tuned LightGBM)')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/formulation_comparison_{_TS}.png', dpi=110); plt.show()
print('Winner (lowest mean White/Black MAE):', res_df.iloc[0]['formulation'])
print(f'Quantile 80% interval (White): coverage {interval_cov:.1%} (target 80%), mean width {interval_width:.0f} Elo')
print(f'Difference-only model: MAE {diff_mae:.1f} vs baseline (gap=0) {baseline_diff_mae:.1f}  '
      f"({'beats' if diff_mae < baseline_diff_mae else 'does NOT beat'} the naive baseline)")
res_df.round(2)